# NIH metrics

This notebook uses information extracted from [NIH Exporter](https://reporter.nih.gov/exporter) to identify NIH-funded users of PhysioNet.

## Setup

In [42]:
import os
import time
from pathlib import Path

import pandas as pd
from tqdm import tqdm
from joblib import Parallel, delayed
from rapidfuzz.distance import JaroWinkler

from twentyfiveyears.nih import (nih_exporter_table_combined, standardize_nih_exporter_project_names,
                                 standardize_nih_exporter_publication_names, get_nih_exporter_project_leader_names,
                                 get_nih_exporter_publication_authors, best_jaro_winkler_match, check_nih_exporter_hits)

In [43]:
# Set the base path
base_path = os.path.join("..", "data")

## Load map of Person IDs

All users are assigned a unique `person_id`.

In [44]:
# Load the map of Person IDs
path = os.path.join(base_path, 'handcrafted', 'person_id_lookup.csv')
df_person_id_mapping = pd.read_csv(path)
df_person_id_mapping.head(3)

,person_id,physionet_id
0,100000000,2
1,100000001,6
2,100000002,8


## Load PhysioNet dataset

Load a dataset containing the list of PhysioNet users

In [45]:
# Load DataFrame of PhysioNet users
path = os.path.join(base_path, 'physionet', 'users.csv')
df_pn_users = pd.read_csv(path, low_memory=False)

In [46]:
df_pn_users.head(3)

,user_id,username,join_date,last_login,registration_ip,is_active_user,primary_email,all_emails,first_names,last_name,...,credentialing_job_title,credentialing_city,credentialing_state_or_province,credentialing_country,credentialing_webpage,credentialing_reference_name,credentialing_reference_email,credentialing_reference_org,credentialing_reference_response,credentialing_research_summary
0,2,ftorres,2019-01-11,2024-05-30 23:51:55.929525+00:00,NaN,True,ftf.dummy@gmail.com,"ftorres@physionet.org, ftf.dummy@gmail.com",Felipe III,Torres Fábregas,...,TEST,TEST,TEST,US,NaN,TEST,felipe.torres.cs@gmail.com,NaN,NaN,TEST
1,6,tompollard,2019-02-19,2024-08-01 15:22:27.844114+00:00,NaN,True,tpollard@mit.edu,"tpollard@mit.edu, tompollardx@gmail.com",Tom,Pollard,...,Research Scientist,Somerville,MA,US,NaN,Tom Pollard,tpollard@mit.edu,NaN,Me.,Laboratory for Computational Physiology
2,8,benjamin,2019-02-21,2024-08-06 20:16:01.888720+00:00,NaN,True,bmoody@mit.edu,"bmoody@mit.edu, benjaminmoody+test123@gmail.co...",Benjamin,Moody,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Now join to the list of Person IDs, and drop unused columns:

In [47]:
# Add person_id column to the users table
df_pn_users = pd.merge(df_pn_users, df_person_id_mapping, left_on='user_id', right_on='physionet_id')

# Only keep the columns we need from the users table
df_pn_users = df_pn_users[['person_id', 'full_name']].copy()

df_pn_users.head(3)

,person_id,full_name
0,100000000,Felipe III Torres Fábregas
1,100000001,Tom Pollard
2,100000002,Benjamin Moody


## Load NIH data

Load data extracted from NIH Exporter

In [35]:
# Set the required variables
nih_exporter_projects_folder = os.path.join(base_path, 'nih', 'exporter', 'projects')
nih_exporter_publications_folder = os.path.join(base_path, 'nih', 'exporter', 'publications')
nih_exporter_start_year = 1995

In [ ]:
# Check to see if PhysioNet users can be found in the NIH exporter data
df_pn_users = check_nih_exporter_hits(df_pn_users, nih_exporter_projects_folder, nih_exporter_publications_folder,nih_exporter_start_year)

In [ ]:
# Rename the 'full_name' column to 'physionet_name'
df_pn_users = df_pn_users.rename(columns={'full_name': 'physionet_name'})

Save the results

In [39]:
# Save the results
save_path = os.path.join(base_path, 'physionet_users_nih_funded.csv')
path = Path(save_path)

# Convert to a path that works on the current OS
normalized_path = path.as_posix() if path.drive else Path(*path.parts).resolve()

# Output the merged DataFrame or save it to a file
df_pn_users.to_csv(normalized_path, index=False)